In [27]:
from diploma.utils import get_contexts, get_qa

import uuid
from tqdm.notebook import tqdm

from diploma.models.octoai import run_chat_completion
from langchain.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain.schema import Document
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import Chroma

In [3]:
import os
from dotenv import load_dotenv

dotenv_path = os.path.join(os.getcwd(), '.env')
load_dotenv(dotenv_path)

token = os.environ["OCTOAI_TOKEN"]
endpoint = os.environ["ENDPOINT"]

In [12]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [4]:
qa = get_qa("datasets/triviaqa_train.json")
contexts = get_contexts("datasets/triviaqa_train.json")

In [5]:
LIMIT = 1000
qa_limit = qa[:LIMIT]
contexts_limit = contexts[:LIMIT]

In [6]:
# Prepare documents
doc_ids = [str(uuid.uuid4()) for _ in contexts_limit]
documents = [Document(page_content=s, metadata={"doc_id": doc_ids[i]}) for i, s in enumerate(contexts_limit)]
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
docs = text_splitter.split_documents(documents)
# Create vectorstore
embedding_function = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
db = Chroma.from_documents(docs, embedding_function)

/home/susbar/Documents/Deelvin/diploma_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
def evaluation_with_context(model_name):
    acc = 0
    print(f"Run evaluation on {model_name} with context")
    for element in qa_limit:
        question, answer = element["question"], element["answer"]
        
        relative_docs = db.similarity_search(question)
        messages = [
        {"role": "system", "content": "Answer only with one word. Use context."},
        {"role": "user", 
        "content": f"Question: {question}\n Context: {relative_docs[0].page_content}"}
        ]
        
        completion = run_chat_completion(model_name, messages, token, endpoint)
        acc += int(answer.lower() in completion["choices"][0]["message"]["content"].lower())

    return acc / len(qa_limit)

def evaluation_without_context(model_name):
    acc = 0
    print(f"Run evaluation on {model_name} without context")
    for element in qa_limit:
        question, answer = element["question"], element["answer"]

        messages = [
        {"role": "system", "content": "Answer only with one word."},
        {"role": "user", "content": f"Question: {question}"}
        ]

        completion = run_chat_completion(model_name, messages, token, endpoint)
        acc += int(answer.lower() in completion["choices"][0]["message"]["content"].lower())

    return acc / len(qa_limit)

In [15]:
acc_llama_70b = evaluation_without_context("llama-2-70b-chat-fp16")
acc_llama_13b = evaluation_without_context("llama-2-13b-chat-fp16")
acc_llama_13b_RAG = evaluation_with_context("llama-2-13b-chat-fp16")

Run evaluation on llama-2-70b-chat-fp16 without context
Run evaluation on llama-2-13b-chat-fp16 without context
Run evaluation on llama-2-13b-chat-fp16 with context


In [16]:
print(f"{LIMIT} examples was used")
print(f"Model: llama-2-70b, acc: {acc_llama_70b}, context: False")
print(f"Model: llama-2-13b, acc: {acc_llama_13b}, context: False")
print(f"Model: llama-2-13b, acc: {acc_llama_13b_RAG}, context: True")

1000 examples was used
Model: llama-2-70b, acc: 0.4, context: False
Model: llama-2-13b, acc: 0.263, context: False
Model: llama-2-13b, acc: 0.346, context: True


In [18]:
doc_ids = [str(uuid.uuid4()) for _ in contexts_limit]
documents = [Document(page_content=s, metadata={"doc_id": doc_ids[i]}) for i, s in enumerate(contexts_limit)]

chunk_sizes = [200, 500, 800, 1000, 1500]
acc_RAG = []
for chunk in chunk_sizes:
    print("Testing chunk size:", chunk)
    text_splitter = CharacterTextSplitter(chunk_size=chunk, chunk_overlap=0)
    docs = text_splitter.split_documents(documents)
    
    embedding_function = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
    db = Chroma.from_documents(docs, embedding_function)
    
    try:
        acc_codellama_7b_RAG = evaluation_with_context("codellama-7b-instruct-fp16")
    except:
        print("Error")
        acc_codellama_7b_RAG = 0.0
    acc_RAG.append(acc_codellama_7b_RAG)    

Testing chunk size: 200
Run evaluation on codellama-7b-instruct-fp16 with context
Testing chunk size: 500
Run evaluation on codellama-7b-instruct-fp16 with context


Exception ignored in: <function tqdm.__del__ at 0x7f4b3858a200>
Traceback (most recent call last):
  File "/home/susbar/Documents/Deelvin/diploma_env/lib/python3.10/site-packages/tqdm/std.py", line 1149, in __del__
    self.close()
  File "/home/susbar/Documents/Deelvin/diploma_env/lib/python3.10/site-packages/tqdm/notebook.py", line 278, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm_notebook' object has no attribute 'disp'


Testing chunk size: 800
Run evaluation on codellama-7b-instruct-fp16 with context
Testing chunk size: 1000
Run evaluation on codellama-7b-instruct-fp16 with context
Testing chunk size: 1500
Run evaluation on codellama-7b-instruct-fp16 with context


In [24]:
import pandas as pd

pd.DataFrame([acc_RAG], columns=chunk_sizes)

,200,500,800,1000,1500
0,0.454,0.444,0.415,0.41,0.41
